In [ ]:
!pip3 install -U neo4j

In [ ]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

In [10]:
# Load the .env file or assign these values manually
load_dotenv()

username = os.getenv("neo4j_username")
password = os.getenv("neo4j_password")

In [13]:
uri = f"neo4j+s://{username}.databases.neo4j.io"

driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
session = driver.session()

print("Successfully connected to Neo4j!")

Successfully connected to Neo4j!


Implementing Order by example from : https://neo4j.com/docs/cypher-manual/5/clauses/order-by/

In [24]:
query1 = """
CREATE
    (o1:Order {id: 'ORD-001', orderDate: datetime('2024-05-01T10:00:00'), total: 550, status: 'shipped'}),
    (o2:Order {id: 'ORD-002', orderDate: datetime('2024-05-02T14:30:00'), total: 1000, status: 'pending'}),
    (o3:Order {id: 'ORD-003', orderDate: datetime('2024-05-03T09:15:00'), total: 550, status: 'pending'}),
    (o4:Order {id: 'ORD-004', orderDate: datetime('2024-05-04T12:45:00'), total: 200}),
    (o5:Order {id: 'ORD-005', orderDate: datetime('2024-05-05T15:00:00'), total: 800, status: 'shipped'}),

    (i1:Item {name: 'Phone', price: 500}),
    (i2:Item {name: 'Laptop', price: 1000}),
    (i3:Item {name: 'Headphones', price: 250}),
    (i4:Item {name: 'Charger', price: 50}),
    (i5:Item {name: 'Keyboard', price: 200}),

    (o1)-[:CONTAINS]->(i1),
    (o1)-[:CONTAINS]->(i4),
    (o2)-[:CONTAINS]->(i2),
    (o3)-[:CONTAINS]->(i1),
    (o3)-[:CONTAINS]->(i4),
    (o4)-[:CONTAINS]->(i5),
    (o5)-[:CONTAINS]->(i1),
    (o5)-[:CONTAINS]->(i3),
    (o5)-[:CONTAINS]->(i4)
"""
result1 = session.run(query1)
print("Orders and items created successfully!")

Orders and items created successfully!


In [ ]:
#Order by total order value
query2 = """
MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total
  ORDER BY total
"""
result2 = session.run(query2)
[(record["order"], record["total"]) for record in result2]

[('ORD-004', 200),
 ('ORD-004', 200),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-005', 800),
 ('ORD-005', 800),
 ('ORD-002', 1000),
 ('ORD-002', 1000)]

In [25]:
query3 = """MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total,
       o.orderDate AS orderDate
  ORDER BY total,
           orderDate"""

result3 = session.run(query3)
[(record["order"], record["total"], record["orderDate"]) for record in result3]

[('ORD-004', 200, neo4j.time.DateTime(2024, 5, 4, 12, 45, 0, 0, tzinfo=<UTC>)),
 ('ORD-001', 550, neo4j.time.DateTime(2024, 5, 1, 10, 0, 0, 0, tzinfo=<UTC>)),
 ('ORD-003', 550, neo4j.time.DateTime(2024, 5, 3, 9, 15, 0, 0, tzinfo=<UTC>)),
 ('ORD-005', 800, neo4j.time.DateTime(2024, 5, 5, 15, 0, 0, 0, tzinfo=<UTC>)),
 ('ORD-002',
  1000,
  neo4j.time.DateTime(2024, 5, 2, 14, 30, 0, 0, tzinfo=<UTC>))]

In [26]:
#This next query counts the number of items contained in each order and then orders the results by the item count.
query4="""MATCH (o:Order)
RETURN o.id AS order,
      COUNT { (o)-[:CONTAINS]->(:Item) } AS itemCount
  ORDER BY itemCount"""

result4 = session.run(query4)
[(record["order"], record["itemCount"]) for record in result4]

[('ORD-002', 1),
 ('ORD-004', 1),
 ('ORD-001', 2),
 ('ORD-003', 2),
 ('ORD-005', 3)]

In [ ]:
#ORDER BY sorts results in an ascending order by default. To explicitly sort results in an ascending order, append ASC[ENDING].
query5="""MATCH (o:Order)
RETURN o.id AS order,
       o.total AS total
  ORDER BY total ASC"""

result5 = session.run(query5)
[(record["order"], record["total"]) for record in result5]

[('ORD-004', 200),
 ('ORD-001', 550),
 ('ORD-003', 550),
 ('ORD-005', 800),
 ('ORD-002', 1000)]

In [28]:
query = ("MATCH (n) DETACH DELETE n")
result = session.run(query)
print("All Nodes and relationships are deleted!")

All Nodes and relationships are deleted!
